# dcgan-wrapper-netG-netD — ex1: wrap two subnets as netG/netD with separate optimizers

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dcgan-wrapper-netG-netD`. Running the final beacon cell reports progress against the `Generative: DCGAN netG+netD wrapper` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: DCGAN netG+netD wrapper` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-wrapper-netG-netD`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-wrapper-netG-netD"
DD_SUBTOPIC = "Generative: DCGAN netG+netD wrapper"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DCGAN wrapper module — quick refresher

ARENA's `DCGAN` class is a *wrapper* — a single `nn.Module` that holds BOTH the generator and the discriminator as submodules:

```python
class DCGAN(nn.Module):
    def __init__(self, ...):
        super().__init__()
        self.netG = Generator(...)
        self.netD = Discriminator(...)
```

**No `forward` method.** Callers invoke `model.netG(noise)` and `model.netD(img)` directly — there isn't a single sensible signature for the joint forward (one takes noise, the other takes images), so we skip it.

**Why bother with a wrapper at all?** Two reasons:
1. `.to(device)` on the wrapper moves both networks in one call.
2. `state_dict()` snapshots BOTH at once for clean checkpointing.

**`parameters()` gotcha.** `dcgan.parameters()` returns ALL params from BOTH subnets — never pass that straight to an optimizer in a GAN. You need TWO optimizers (`Adam(dcgan.netG.parameters(), ...)` and `Adam(dcgan.netD.parameters(), ...)`) so the gradient steps stay adversarial.

### Exercise 1 — wrap two subnets as netG/netD with separate optimizers

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Create
> LO: Create a wrapper `nn.Module` that holds `netG` and `netD` as submodules (no joint `forward`) and verify both subnets are reachable for separate optimizer construction.
> Keywords: dcgan, wrapper-module, submodules, two-optimizers
> ```

**KCs targeted:** `wrapper-holds-netG-netD`, `wrapper-no-joint-forward`

Implement `ex1_make_dcgan_wrapper(generator, discriminator)`. The DCGAN container-module pattern that ARENA's part-5 GAN training loop assumes:

1. The function takes two ALREADY-CONSTRUCTED `nn.Module` instances — `generator` and `discriminator`.
2. Return an instance of a `nn.Module` subclass that has:
   - `wrapper.netG` set to `generator` (auto-registered as submodule).
   - `wrapper.netD` set to `discriminator` (auto-registered as submodule).
   - NO `forward` method — callers must invoke `wrapper.netG(noise)` or `wrapper.netD(img)` directly.
3. `super().__init__()` must run BEFORE assigning `netG` / `netD` — otherwise the assignment raises an `AttributeError`.

Input: two `nn.Module` instances.
Output: a single `nn.Module` instance whose `.netG` and `.netD` attributes are the input modules.

The visualization renders a parameter-count bar chart for both subnets to confirm the wrapper exposes them as distinct trainable submodules.

In [ ]:
def ex1_make_dcgan_wrapper(generator: nn.Module, discriminator: nn.Module) -> nn.Module:
    """Build a wrapper that holds netG and netD as submodules."""
    raise NotImplementedError()


def _test_ex1():
    from torch import nn

    # Build two toy subnets.
    gen = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
    disc = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    wrapper = ex1_make_dcgan_wrapper(gen, disc)

    # Wrapper is an nn.Module.
    assert isinstance(wrapper, nn.Module), f'expected nn.Module, got {type(wrapper)}'

    # netG / netD are the SAME instances we passed in (identity, not a copy).
    assert wrapper.netG is gen, 'wrapper.netG must be the SAME generator instance'
    assert wrapper.netD is disc, 'wrapper.netD must be the SAME discriminator instance'

    # Both subnets are auto-registered (visible via named_children).
    child_names = dict(wrapper.named_children())
    assert 'netG' in child_names, f'netG not registered as submodule; named_children={list(child_names)}'
    assert 'netD' in child_names, f'netD not registered as submodule; named_children={list(child_names)}'

    # Wrapper.parameters() walks BOTH subnets.
    n_params_total = sum(p.numel() for p in wrapper.parameters())
    n_params_gen = sum(p.numel() for p in gen.parameters())
    n_params_disc = sum(p.numel() for p in disc.parameters())
    assert n_params_total == n_params_gen + n_params_disc, (
        f'wrapper params {n_params_total} != gen {n_params_gen} + disc {n_params_disc}'
    )

    # Construct the TWO separate optimizers the GAN loop needs.
    opt_g = t.optim.Adam(wrapper.netG.parameters(), lr=1e-3)
    opt_d = t.optim.Adam(wrapper.netD.parameters(), lr=1e-3)
    assert opt_g.param_groups[0]['params'][0] is next(gen.parameters()), 'opt_g must train netG params'
    assert opt_d.param_groups[0]['params'][0] is next(disc.parameters()), 'opt_d must train netD params'

    # Wrapper does NOT define a joint forward — calling wrapper(x) on a tensor of
    # arbitrary shape should fail (since base nn.Module has no forward).
    try:
        wrapper(t.zeros(2, 8))
        raise AssertionError('wrapper(x) must NOT work — wrapper has no joint forward')
    except (NotImplementedError, AttributeError, TypeError):
        pass  # expected — no forward defined

    # Subnets work individually through the wrapper attributes.
    noise = t.randn(3, 8)
    fake = wrapper.netG(noise)
    assert fake.shape == (3, 4), f'netG output shape wrong: {tuple(fake.shape)}'
    score = wrapper.netD(fake)
    assert score.shape == (3, 1), f'netD output shape wrong: {tuple(score.shape)}'

    # .to(dtype=float64) on wrapper moves BOTH subnets.
    wrapper.to(t.float64)
    assert next(wrapper.netG.parameters()).dtype == t.float64, '.to() must move netG params'
    assert next(wrapper.netD.parameters()).dtype == t.float64, '.to() must move netD params'
    wrapper.to(t.float32)  # restore for downstream cells

    # --- Visualization: per-subnet param-count bar chart ---
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(['netG', 'netD'], [n_params_gen, n_params_disc],
           color=['steelblue', 'coral'], edgecolor='black')
    ax.set_ylabel('parameter count')
    ax.set_title('ex1 wrapper exposes netG + netD as distinct trainable subnets')
    for i, n in enumerate([n_params_gen, n_params_disc]):
        ax.text(i, n, str(n), ha='center', va='bottom')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_make_dcgan_wrapper(generator, discriminator):
    from torch import nn
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()       # MUST be first — wires up _modules dict
            self.netG = netG
            self.netD = netD
        # No forward method — see solution notes.
    return DCGAN(generator, discriminator)
```

**Why no joint `forward`.** A GAN's generator takes noise `(B, latent_dim)`; the discriminator takes images `(B, C, H, W)`. There isn't a single tensor signature that makes sense for both, so we don't define one. Callers always invoke `model.netG(noise)` or `model.netD(img)` explicitly.

**Why a wrapper at all.** Three reasons:
1. `model.to(device)` moves both subnets in one call (the test demonstrated this with `to(float64)`).
2. `model.state_dict()` snapshots both for clean checkpointing.
3. `model.train()` / `model.eval()` toggles both at once — important for BatchNorm and Dropout layers.

**Two optimizers, one model.** The GAN training step alternates: update netD on a batch (with `opt_d`), then update netG on a batch (with `opt_g`). NEVER pass `model.parameters()` directly to an optimizer — that would couple the two networks into one gradient step, breaking the adversarial dynamic.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()